In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
import torch

from torch import Tensor
from tsdm.encoders import time2float
from torch.utils.data import SequentialSampler, Dataset, DataLoader
rng = np.random.default_rng()
np.set_printoptions()

In [3]:
from tsdm.datasets import ETTh1
ds = ETTh1.dataset

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


In [4]:
train_dataset = ds[:"2017-06-30"]  # inclusive range!
valid_dataset = ds["2017-07-01":"2017-10-31"]  # inclusive range!
trial_dataset = ds["2017-11-01":"2018-02-28"]  # inclusive range!

forecasting_horizon: Literal[24, 48, 168, 336, 960] = 24,
observation_horizon: Literal[24, 48, 96, 168, 336, 720] = 96,
test_metric: Literal["MSE", "MAE"] = "MSE",

target = "OT"

'OT'

In [5]:
from tsdm.datasets import SequenceDataset
from tsdm.util.samplers import SequenceSampler

In [6]:
time_encoder = time2float
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

train_dataset.drop(columns="OT")

_T = time_encoder(train_dataset.index)
_X = train_dataset.drop(columns="OT").values
_Y = train_dataset["OT"].values

T = torch.tensor(_T, device=device, dtype=dtype)
X = torch.tensor(_X, device=device, dtype=dtype)
Y = torch.tensor(_Y, device=device, dtype=dtype)

tensor([30.5310, 27.7870, 27.7870,  ..., 17.5160, 17.8680, 18.1500],
       device='cuda:0')

In [7]:
class SequenceDataset(Dataset):
    def __init__(self, tensors: list[Tensor]):
        assert all( len(x) == len(tensors[0]) for x in tensors)
        self.tensors = tensors
    
    def __len__(self):
        return len(self.tensors[0])
        
    def __getitem__(self, idx):
        return [x[idx] for x in self.tensors]
    
class SequenceSampler(torch.utils.data.Sampler):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
        
    def __iter__(self):
        print(len(self.data), self.seq_len)
        for idx in range(len(self.data) - self.seq_len):
            yield range(idx, idx+self.seq_len)

In [8]:
train_dataset = SequenceDataset([T, X, Y])
sampler = SequenceSampler(train_dataset, 2)
last = list(iter(DataLoader(train_dataset, sampler=sampler)))[-1]

8760 2


[tensor([[8757., 8758.]], device='cuda:0'),
 tensor([[[ 6.6310,  0.0000,  3.6960,  0.2490,  2.8940, -0.5790],
          [ 7.1670,  0.0000,  4.0150,  0.3910,  2.8630, -0.5790]]],
        device='cuda:0'),
 tensor([[17.5160, 17.8680]], device='cuda:0')]

# Implemented Task

In [9]:
from tsdm.tasks import ETDatasetInformer

In [10]:
task = ETDatasetInformer()

In [11]:
dloader = task.get_dataloader("test")

In [12]:
for item in dloader:
    t, x, y = item
    print(t,x,y)

tensor([[  0.,   1.,   2.,  ..., 117., 118., 119.],
        [  1.,   2.,   3.,  ..., 118., 119., 120.],
        [  2.,   3.,   4.,  ..., 119., 120., 121.],
        ...,
        [ 29.,  30.,  31.,  ..., 146., 147., 148.],
        [ 30.,  31.,  32.,  ..., 147., 148., 149.],
        [ 31.,  32.,  33.,  ..., 148., 149., 150.]], device='cuda:0') tensor([[[ 11.9890,   4.6220,   9.5940,   2.7360,   2.3150,   1.0660],
         [ 11.9890,   4.0190,   9.4170,   2.4160,   2.2840,   1.1880],
         [ 10.6500,   3.5500,   8.2440,   1.9190,   2.1930,   1.2490],
         ...,
         [  9.4440,   2.1430,   6.7870,   0.7820,   2.7110,   1.3100],
         [  9.4440,   2.4780,   7.2490,   1.2440,   2.2240,   1.1880],
         [ 10.5160,   3.6170,   7.3560,   1.8480,   2.4980,   1.2180]],

        [[ 11.9890,   4.0190,   9.4170,   2.4160,   2.2840,   1.1880],
         [ 10.6500,   3.5500,   8.2440,   1.9190,   2.1930,   1.2490],
         [ 10.5160,   3.5500,   8.5640,   2.1680,   2.0710,   1.1270],
  